# 02 Event study (dossier §7.6)

Before any machine learning, check that there is something to measure.

1. Mean CAR[0, 2] by sign of the shock (negative, neutral, positive), Student t-test and sign test.
2. Anticipation check: mean CAR[-5, -1] (§6.5, rule 4).
3. Media versus control companies: is the mean |CAR| larger for media figures?
4. Mean cumulative abnormal return from J-10 to J+10 by sign: the classic event-study figure.

Kept events only, train and validation splits by default: the test period (2024-2025) stays untouched until the final evaluation (§9.5), and natural experiments are reserved for §11.2.

Set `KPR_USE_FIXTURES=1` before starting Jupyter to run on the synthetic fixtures. All logic lives in `kpr.validation.event_study`; this notebook only calls it. Do not commit outputs over 1 MB.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from kpr.utils.io import read_table, use_fixtures
from kpr.validation import event_study as ES

SPLITS = ("train", "validation")  # never add "test" before the final evaluation (task M12)
print("fixtures" if use_fixtures() else "real data")

In [ ]:
events_all = read_table("events")
events = ES.select(events_all, SPLITS)
print(f"{len(events_all)} events, {len(events)} kept in {SPLITS}")
events_all.groupby("split")["excluded"].agg(["size", "sum"]).rename(
    columns={"size": "events", "sum": "excluded"})

## Exclusions (§7.4)

Excluded shocks are kept and counted, to check that the filters do not create a bias.

In [ ]:
reasons = events_all["exclusion_reason"].dropna().str.split(";").explode()
reasons.value_counts().rename("events")

## 1. Mean CAR[0, 2] by sign

In [ ]:
ES.mean_car_table(events, by="sign", car="car_0_2").round(4)

## 2. Anticipation: CAR[-5, -1] by sign

A systematically strong pre-event CAR would mean the news was known before the attention peak.

In [ ]:
ES.mean_car_table(events, by="sign", car="car_m5_m1").round(4)

## 3. Media versus control companies (|CAR[0, 2]|)

In [ ]:
groups = ES.company_groups()
if not groups:
    print("universe.yaml has no group yet (task D1): comparison skipped")
else:
    display(pd.Series(ES.compare_groups(events, groups)))

## 4. Mean CAR from J-10 to J+10 by sign

In [ ]:
prices, market = read_table("prices"), read_table("market")
paths = ES.car_paths(events, prices, market, window=(-10, 10))
summary = ES.mean_paths(paths, events.set_index("event_id")["sign"])
ax = ES.plot_mean_paths(summary, title="Mean CAR around attention shocks, by sign")
plt.show()

## Reading the results

- If no group shows an effect, that is a result to discuss; the models will then look for heterogeneous effects (some persons only).
- If an effect shows, the ML part explains and personalises it (§7.6).